# 06 · Add your own model (the runner engine)

Every **runner tier** — `sequence_embedding`, `structure_prediction`,
`conformer_generation`, `binding_prediction`, `docking`, `scoring` — shares one engine
(`kinapse.runners`). Adding a model is the same everywhere and needs no kinapse fork.
Full recipe: [`../docs/ADDING_A_MODEL.md`](../docs/ADDING_A_MODEL.md).

## See what's registered

In [ ]:
from kinapse import runners
for tier in ["structure_prediction", "binding_prediction", "docking", "scoring"]:
    print(f"{tier:22s}", [s.name for s in runners.specs(tier)])

## Register your own — a native runner
A model is a `RunnerSpec` + a runner. The quickest is a `native` backend: an in-process
`func(inputs: dict) -> dict` (a `status` key is added for you; a raised error is captured,
never crashing a batch).

In [ ]:
from kinapse.runners import RunnerSpec, register

def _predict(inputs: dict) -> dict:
    # your model here — this stand-in just echoes
    return {"score": len(inputs.get("cdr3b", "")) / 20.0}

register(RunnerSpec(name="my_predictor", tier="binding_prediction", backend="native",
                    status="stable", outputs=("score",), func=_predict), replace=True)

In [ ]:
import kinapse.binding_prediction as bp
print("registered:", [s.name for s in bp.available() if s.name == "my_predictor"])
bp.run("my_predictor", {"cdr3b": "CASSPLGREGLNTEAF"})

## Isolated (`uvenv`) runners & shipping in your own package
For heavy/conflicting deps, use a `uvenv` runner — a JSON-in/JSON-out script that kinapse
executes in a throwaway env via `uv`, so its dependencies never touch kinapse:

```python
register(RunnerSpec(name="my_gpu_model", tier="binding_prediction", backend="uvenv",
                    requirements=("torch==2.3", "my-model-pkg"), python="3.11",
                    runner="/abs/path/to/my_runner.py", outputs=("score",)))
```

And you can register from **your own** pip package (no fork) via an entry point — kinapse
auto-discovers it once installed:

```toml
# your package's pyproject.toml
[project.entry-points."kinapse.runners"]
my_models = "my_pkg.kinapse_plugin:register_all"
```
See [`../docs/ADDING_A_MODEL.md`](../docs/ADDING_A_MODEL.md) for the full contract.